In [1]:
import sys
import json
from pathlib import Path

# Add the llm-vulvariant source to the path
sys.path.insert(0, '~/vuln/llm-vulvariant/src')

from profiler.software.models import SoftwareProfile, ModuleInfo
from similarity.embedding_retrieval import EmbeddingRetriever, EmbeddingRetrievalConfig

## Step 1: Load Software Profiles

In [2]:
# Load ms-swift profile
ms_swift_profile_path = Path('/home/dongtian/vuln/llm-vulvariant/repo-profiles-ds/ms-swift/2c19674fe612d0b94fe671789d8a87a594838db0/software_profile.json')
with open(ms_swift_profile_path, 'r', encoding='utf-8') as f:
    ms_swift_data = json.load(f)
ms_swift_profile = SoftwareProfile.from_dict(ms_swift_data)
print(f"Loaded ms-swift profile: {ms_swift_profile.name}")
print(f"  Version: {ms_swift_profile.version}")
print(f"  Modules: {len(ms_swift_profile.modules)}")

# Load LLaMA-Factory profile
llama_factory_profile_path = Path('/home/dongtian/vuln/llm-vulvariant/repo-profiles-ds/LLaMA-Factory/9f73a6eb234fe3df67d9d921c157fde4a0faca6a/software_profile.json')
with open(llama_factory_profile_path, 'r', encoding='utf-8') as f:
    llama_factory_data = json.load(f)
llama_factory_profile = SoftwareProfile.from_dict(llama_factory_data)
print(f"\nLoaded LLaMA-Factory profile: {llama_factory_profile.name}")
print(f"  Version: {llama_factory_profile.version}")
print(f"  Modules: {len(llama_factory_profile.modules)}")

Loaded ms-swift profile: ms-swift
  Version: 2c19674fe612d0b94fe671789d8a87a594838db0
  Modules: 11

Loaded LLaMA-Factory profile: LLaMA-Factory
  Version: 9f73a6eb234fe3df67d9d921c157fde4a0faca6a
  Modules: 15


## Step 2: Load Text Embedding Model

In [3]:
# Load text embedding model for general text comparison
text_embedding_model_path = Path('/home/dongtian/vuln/models/BAAI--bge-large-en-v1.5')
text_config = EmbeddingRetrievalConfig(
    model_path=text_embedding_model_path,
    device='cuda:7',  # Use CPU to avoid OOM
    batch_size=8,
    normalize=True
)
text_retriever = EmbeddingRetriever(config=text_config)
print(f"Text embedding model loaded from: {text_embedding_model_path}")

Text embedding model loaded from: /home/dongtian/vuln/models/BAAI--bge-large-en-v1.5


## Step 3: Compare Text-Related Information

We'll compare:
- Description
- Target applications
- Target users

In [5]:
# Compare descriptions
desc_sim = text_retriever.similarity(ms_swift_profile.description, llama_factory_profile.description)
print(f"Description similarity: {desc_sim:.4f}")
print(f"\nms-swift: {ms_swift_profile.description[:200]}...")
print(f"\nLLaMA-Factory: {llama_factory_profile.description[:200]}...")

# Compare target applications
ms_swift_apps = " | ".join(ms_swift_profile.target_application)
llama_apps = " | ".join(llama_factory_profile.target_application)
apps_sim = text_retriever.similarity(ms_swift_apps, llama_apps)
print(f"\n\nTarget applications similarity: {apps_sim:.4f}")

# Compare target users
ms_swift_users = " | ".join(ms_swift_profile.target_user)
llama_users = " | ".join(llama_factory_profile.target_user)
users_sim = text_retriever.similarity(ms_swift_users, llama_users)
print(f"Target users similarity: {users_sim:.4f}")

print(f"\n=== Overall Text Similarity Score: {(desc_sim + apps_sim + users_sim) / 3:.4f} ===")

Description similarity: 0.8365

ms-swift: ms-swift (SWIFT) is a comprehensive framework for fine-tuning and deploying large language models and multimodal models, offering scalable infrastructure for model adaptation across training, inferenc...

LLaMA-Factory: LLaMA Factory is a unified framework for efficiently fine-tuning and customizing 100+ large language models (LLMs) and multimodal models with zero-code CLI and Web UI interfaces. It provides comprehen...


Target applications similarity: 0.8529
Target users similarity: 0.9063

=== Overall Text Similarity Score: 0.8652 ===


## Step 4: Find Similar Modules Using Embedding-Based Approach

We'll use text embeddings to find similar modules by comparing module names and descriptions in a fuzzy, error-tolerant way.

In [6]:
import numpy as np
from typing import List, Dict, Tuple

def create_module_text(module: ModuleInfo) -> str:
    """Create a comprehensive text representation of a module for embedding."""
    parts = [
        f"Module: {module.name}",
        f"Category: {module.category}",
        f"Description: {module.description}",
    ]
    
    if module.public_apis:
        parts.append(f"APIs: {', '.join(module.public_apis[:5])}")
    
    if module.external_dependencies:
        parts.append(f"Dependencies: {', '.join(module.external_dependencies[:5])}")
    
    if module.data_formats:
        parts.append(f"Data formats: {', '.join(module.data_formats)}")
    
    return " | ".join(parts)

# Create text representations for all modules
print("Creating text representations for modules...")
ms_swift_module_texts = [create_module_text(m) for m in ms_swift_profile.modules if isinstance(m, ModuleInfo)]
llama_module_texts = [create_module_text(m) for m in llama_factory_profile.modules if isinstance(m, ModuleInfo)]

print(f"ms-swift modules: {len(ms_swift_module_texts)}")
print(f"LLaMA-Factory modules: {len(llama_module_texts)}")

# Generate embeddings for all modules
print("\nGenerating embeddings...")
ms_swift_embeddings = text_retriever.embed(ms_swift_module_texts)
llama_embeddings = text_retriever.embed(llama_module_texts)

print(f"ms-swift embeddings shape: {len(ms_swift_embeddings)}x{len(ms_swift_embeddings[0])}")
print(f"LLaMA-Factory embeddings shape: {len(llama_embeddings)}x{len(llama_embeddings[0])}")

Creating text representations for modules...
ms-swift modules: 11
LLaMA-Factory modules: 15

Generating embeddings...
ms-swift embeddings shape: 11x1024
LLaMA-Factory embeddings shape: 15x1024


In [7]:
# Find similar module pairs using cosine similarity
def find_similar_modules(
    source_modules: List[ModuleInfo],
    target_modules: List[ModuleInfo],
    source_embeddings: List[List[float]],
    target_embeddings: List[List[float]],
    threshold: float = 0.7
) -> List[Dict[str, any]]:
    """
    Find similar module pairs based on embedding similarity.
    
    Args:
        source_modules: Source modules list
        target_modules: Target modules list
        source_embeddings: Embeddings for source modules
        target_embeddings: Embeddings for target modules
        threshold: Similarity threshold (0-1)
    
    Returns:
        List of similar module pairs with similarity scores
    """
    similar_pairs = []
    
    for i, (source_mod, source_emb) in enumerate(zip(source_modules, source_embeddings)):
        if not isinstance(source_mod, ModuleInfo):
            continue
            
        best_match = None
        best_score = threshold
        
        for j, (target_mod, target_emb) in enumerate(zip(target_modules, target_embeddings)):
            if not isinstance(target_mod, ModuleInfo):
                continue
            
            # Calculate cosine similarity (embeddings are already normalized)
            similarity = sum(a * b for a, b in zip(source_emb, target_emb))
            
            if similarity > best_score:
                best_score = similarity
                best_match = (j, target_mod)
        
        if best_match is not None:
            similar_pairs.append({
                'source_idx': i,
                'source_module': source_mod,
                'target_idx': best_match[0],
                'target_module': best_match[1],
                'similarity': best_score
            })
    
    return sorted(similar_pairs, key=lambda x: x['similarity'], reverse=True)

# Filter ModuleInfo objects
ms_swift_modules = [m for m in ms_swift_profile.modules if isinstance(m, ModuleInfo)]
llama_modules = [m for m in llama_factory_profile.modules if isinstance(m, ModuleInfo)]

# Find similar modules
print("Finding similar module pairs...")
similar_pairs = find_similar_modules(
    ms_swift_modules,
    llama_modules,
    ms_swift_embeddings,
    llama_embeddings,
    threshold=0.7
)

print(f"\nFound {len(similar_pairs)} similar module pairs (threshold=0.7)")
print("\nTop 10 similar module pairs:")
print("="*100)
for i, pair in enumerate(similar_pairs[:10], 1):
    print(f"\n{i}. Similarity: {pair['similarity']:.4f}")
    print(f"   ms-swift: {pair['source_module'].name}")
    print(f"   LLaMA-Factory: {pair['target_module'].name}")
    if pair['source_module'].category or pair['target_module'].category:
        print(f"   Categories: {pair['source_module'].category} <-> {pair['target_module'].category}")

Finding similar module pairs...

Found 11 similar module pairs (threshold=0.7)

Top 10 similar module pairs:

1. Similarity: 0.8875
   ms-swift: UI
   LLaMA-Factory: Web UI

2. Similarity: 0.8322
   ms-swift: Utils
   LLaMA-Factory: Utilities

3. Similarity: 0.8159
   ms-swift: CLI
   LLaMA-Factory: CLI Launcher

4. Similarity: 0.8018
   ms-swift: Trainers
   LLaMA-Factory: Training Algorithms

5. Similarity: 0.7888
   ms-swift: LLM Core
   LLaMA-Factory: Data Processing

6. Similarity: 0.7795
   ms-swift: Hub
   LLaMA-Factory: Model Management

7. Similarity: 0.7724
   ms-swift: Megatron
   LLaMA-Factory: Training Orchestration

8. Similarity: 0.7693
   ms-swift: Plugin
   LLaMA-Factory: Hyperparameter Management

9. Similarity: 0.7458
   ms-swift: Ray
   LLaMA-Factory: Utilities

10. Similarity: 0.7259
   ms-swift: Version
   LLaMA-Factory: Legacy v1


## Step 5: Load Code Embedding Model

Now we'll load the code-specific embedding model to compare actual code within similar modules.

In [9]:
# Load code embedding model
code_embedding_model_path = Path('/home/dongtian/vuln/models/jinaai--jina-code-embeddings-1.5b')
code_config = EmbeddingRetrievalConfig(
    model_path=code_embedding_model_path,
    device='cuda:7',  # Use CPU to avoid OOM
    batch_size=4,  # Smaller batch size for larger model
    max_length=2048,  # Longer context for code
    normalize=True
)
code_retriever = EmbeddingRetriever(config=code_config)
print(f"Code embedding model loaded from: {code_embedding_model_path}")

Code embedding model loaded from: /home/dongtian/vuln/models/jinaai--jina-code-embeddings-1.5b


## Case Study: Hub (ms-swift) vs Model Management (LLaMA-Factory)

Deep dive into the similarity between two highly related modules across different repositories.

### Refactored Case Study Function

This function performs a comprehensive similarity analysis between two modules from different repositories.

In [30]:
from itertools import product
def compare_modules_detailed(
    module1_name: str,
    module2_name: str,
    profile1: SoftwareProfile,
    profile2: SoftwareProfile,
    repo1_path: str,
    repo2_path: str,
    text_retriever: EmbeddingRetriever,
    code_retriever: EmbeddingRetriever,
    max_files: int = 10
) -> dict:
    """
    Perform a comprehensive similarity analysis between two modules from different repositories.
    
    Args:
        module1_name: Name of the first module
        module2_name: Name of the second module
        profile1: SoftwareProfile of the first repository
        profile2: SoftwareProfile of the second repository
        repo1_path: File system path to the first repository
        repo2_path: File system path to the second repository
        text_retriever: EmbeddingRetriever for text comparison
        code_retriever: EmbeddingRetriever for code comparison
        max_files: Maximum number of code files to analyze per module
    
    Returns:
        Dictionary containing all comparison results
    """
    import re
    from collections import Counter
    
    results = {
        'module1_name': module1_name,
        'module2_name': module2_name,
        'found': False,
        'metadata': {},
        'api_similarity': {},
        'code_similarity': {},
        'pattern_analysis': {},
        'overall_scores': {}
    }
    
    # Find modules
    module1 = None
    module2 = None
    
    modules1 = [m for m in profile1.modules if isinstance(m, ModuleInfo)]
    modules2 = [m for m in profile2.modules if isinstance(m, ModuleInfo)]
    
    for module in modules1:
        if module.name == module1_name:
            module1 = module
            break
    
    for module in modules2:
        if module.name == module2_name:
            module2 = module
            break
    
    if not module1 or not module2:
        results['error'] = f"Module not found: {module1_name if not module1 else module2_name}"
        return results
    
    results['found'] = True
    
    # ========== 1. METADATA COMPARISON ==========
    print("\n" + "="*100)
    print(f"MODULE COMPARISON: {module1_name} vs {module2_name}")
    print("="*100)
    
    results['metadata'] = {
        'module1': {
            'name': module1.name,
            'category': module1.category,
            'description': module1.description,
            'file_count': len(module1.files),
            'public_apis_count': len(module1.public_apis),
            'external_deps_count': len(module1.external_dependencies)
        },
        'module2': {
            'name': module2.name,
            'category': module2.category,
            'description': module2.description,
            'file_count': len(module2.files),
            'public_apis_count': len(module2.public_apis),
            'external_deps_count': len(module2.external_dependencies)
        }
    }
    
    # Description similarity
    desc_similarity = text_retriever.similarity(module1.description, module2.description)
    results['metadata']['description_similarity'] = desc_similarity
    
    print(f"\n{'Attribute':<25} {module1_name:<40} {module2_name:<40}")
    print("-"*100)
    print(f"{'Category':<25} {module1.category:<40} {module2.category:<40}")
    print(f"{'File Count':<25} {len(module1.files):<40} {len(module2.files):<40}")
    print(f"{'Description Similarity':<25} {desc_similarity:.4f}")
    
    # Dependency overlap
    deps1 = set(module1.external_dependencies)
    deps2 = set(module2.external_dependencies)
    common_deps = deps1 & deps2
    
    results['metadata']['dependency_overlap'] = {
        'module1_count': len(deps1),
        'module2_count': len(deps2),
        'common_count': len(common_deps),
        'common_deps': list(common_deps)
    }
    
    print(f"\nShared dependencies: {len(common_deps)}/{max(len(deps1), len(deps2))}")
    
    # ========== 2. FIND SIMILAR FUNCTIONS ==========
    print("\n" + "-"*100)
    print("SIMILAR FUNCTIONS ANALYSIS")
    print("-"*100)
    
    # Extract functions from both modules
    def extract_functions_from_files(module: ModuleInfo, repo_path: str, max_files: int = 10) -> list:
        """Extract function definitions with their file paths."""
        functions = []
        for file_path in module.files[:max_files]:
            full_path = os.path.join(repo_path, file_path)
            if os.path.exists(full_path) and full_path.endswith('.py'):
                try:
                    with open(full_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read()
                        # Extract function names
                        func_pattern = r'def\s+([a-zA-Z_][a-zA-Z0-9_]*)\s*\('
                        func_names = re.findall(func_pattern, content)
                        for func_name in func_names:
                            functions.append({
                                'name': func_name,
                                'file': file_path
                            })
                except Exception as e:
                    continue
        return functions
    
    # Extract functions from both modules
    funcs1 = extract_functions_from_files(module1, repo1_path, max_files)
    funcs2 = extract_functions_from_files(module2, repo2_path, max_files)
    
    print(f"Extracted {len(funcs1)} functions from {module1_name}")
    print(f"Extracted {len(funcs2)} functions from {module2_name}")
    
    # Find similar functions using embeddings
    similar_functions = []
    
    if funcs1 and funcs2:
        # Get unique function names for embedding
        func1_names = [f['name'] for f in funcs1]
        func2_names = [f['name'] for f in funcs2]
        
        # Generate embeddings (limit to avoid memory issues)
        max_funcs = 50
        func1_sample = funcs1[:max_funcs]
        func2_sample = funcs2[:max_funcs]
        
        func1_names_sample = [f['name'] for f in func1_sample]
        func2_names_sample = [f['name'] for f in func2_sample]
        
        if func1_names_sample and func2_names_sample:
            func1_embeddings = text_retriever.embed(func1_names_sample)
            func2_embeddings = text_retriever.embed(func2_names_sample)
            
            # Find similar function pairs
            for i, (func1, emb1) in enumerate(zip(func1_sample, func1_embeddings)):
                best_match = None
                best_score = 0.5  # Higher threshold for function names
                
                for j, (func2, emb2) in enumerate(zip(func2_sample, func2_embeddings)):
                    # Skip if exact match (already covered)
                    if func1['name'] == func2['name']:
                        continue
                    
                    similarity = sum(a * b for a, b in zip(emb1, emb2))
                    if similarity > best_score:
                        best_score = similarity
                        best_match = (j, func2, similarity)
                
                if best_match is not None:
                    similar_functions.append({
                        'module1_func': func1['name'],
                        'module1_file': func1['file'],
                        'module2_func': best_match[1]['name'],
                        'module2_file': best_match[1]['file'],
                        'similarity': best_match[2]
                    })
        
        # Also find exact matches
        exact_matches = []
        func1_dict = {}
        for f in funcs1:
            if f['name'] not in func1_dict:
                func1_dict[f['name']] = []
            func1_dict[f['name']].append(f['file'])
        
        for f2 in funcs2:
            if f2['name'] in func1_dict:
                for f1_file in func1_dict[f2['name']]:
                    exact_matches.append({
                        'func_name': f2['name'],
                        'module1_file': f1_file,
                        'module2_file': f2['file']
                    })
    
    # Store results
    results['similar_functions'] = {
        'exact_matches': exact_matches,
        'similar_matches': sorted(similar_functions, key=lambda x: x['similarity'], reverse=True)[:20],
        'exact_count': len(exact_matches),
        'similar_count': len(similar_functions)
    }
    
    # Print results
    print(f"\nFound {len(exact_matches)} exact function name matches")
    if exact_matches:
        print("\nExact matches (showing first 10):")
        for i, match in enumerate(exact_matches[:10], 1):
            print(f"  {i}. {match['func_name']}")
            print(f"     {module1_name}: {match['module1_file']}")
            print(f"     {module2_name}: {match['module2_file']}")
    
    print(f"\nFound {len(similar_functions)} similar function name pairs (threshold=0.75)")
    if similar_functions:
        similar_sorted = sorted(similar_functions, key=lambda x: x['similarity'], reverse=True)
        print("\nTop 10 similar function pairs:")
        for i, match in enumerate(similar_sorted[:10], 1):
            print(f"  {i}. Similarity: {match['similarity']:.4f}")
            print(f"     {module1_name}: {match['module1_func']:<30} ({match['module1_file']})")
            print(f"     {module2_name}: {match['module2_func']:<30} ({match['module2_file']})")
    
    return results

### Example Usage

In [31]:
# Example 1:
results_hub_vs_mgmt = compare_modules_detailed(
    module1_name="LLM Core",
    module2_name="Data Processing",
    profile1=ms_swift_profile,
    profile2=llama_factory_profile,
    repo1_path=ms_swift_repo_path,
    repo2_path=llama_factory_repo_path,
    text_retriever=text_retriever,
    code_retriever=code_retriever,
    max_files=10
)


MODULE COMPARISON: LLM Core vs Data Processing

Attribute                 LLM Core                                 Data Processing                         
----------------------------------------------------------------------------------------------------
Category                                                                                                   
File Count                262                                      11                                      
Description Similarity    0.6885

Shared dependencies: 24/100

----------------------------------------------------------------------------------------------------
SIMILAR FUNCTIONS ANALYSIS
----------------------------------------------------------------------------------------------------
Extracted 282 functions from LLM Core
Extracted 165 functions from Data Processing

Found 44 exact function name matches

Exact matches (showing first 10):
  1. __post_init__
     LLM Core: swift/llm/dataset/loader.py
     Data Proces

### Access Similar Functions Results

In [32]:
# Access the similar functions results
print("="*100)
print("SIMILAR FUNCTIONS ANALYSIS RESULTS (Excluding common function names)")
print("="*100)

# Define common function names to ignore
COMMON_FUNCTIONS = {
    '__init__', '__call__', '__post_init__', '__str__', '__repr__', '__eq__', 
    '__hash__', '__len__', '__getitem__', '__setitem__', '__delitem__',
    '__iter__', '__next__', '__enter__', '__exit__', '__del__', '__new__',
    '__bool__', '__contains__', '__add__', '__sub__', '__mul__', '__div__',
    'get', 'set', 'update', 'save', 'load', 'run', 'execute', 'process'
}

similar_funcs = results_hub_vs_mgmt['similar_functions']

# Filter exact matches
exact_matches_filtered = [
    m for m in similar_funcs['exact_matches'] 
    if m['func_name'] not in COMMON_FUNCTIONS
]

# Filter similar matches
similar_matches_filtered = [
    m for m in similar_funcs['similar_matches']
    if m['module1_func'] not in COMMON_FUNCTIONS and m['module2_func'] not in COMMON_FUNCTIONS
]

print(f"\n1. Summary:")
print(f"   - Total exact matches: {similar_funcs['exact_count']}")
print(f"   - Filtered exact matches (excluding common names): {len(exact_matches_filtered)}")
print(f"   - Total similar pairs: {similar_funcs['similar_count']}")
print(f"   - Filtered similar pairs (excluding common names): {len(similar_matches_filtered)}")

if exact_matches_filtered:
    print(f"\n2. Exact Matches (first 10):")
    for i, match in enumerate(exact_matches_filtered[:10], 1):
        print(f"   {i}. Function: {match['func_name']}")
        print(f"      LLM Core:        {match['module1_file']}")
        print(f"      Data Processing: {match['module2_file']}")
else:
    print(f"\n2. No significant exact matches after filtering common functions")

if similar_matches_filtered:
    print(f"\n3. Similar Matches (first 10):")
    for i, match in enumerate(similar_matches_filtered[:10], 1):
        print(f"   {i}. Similarity: {match['similarity']:.4f}")
        print(f"      LLM Core:        {match['module1_func']:<25} in {match['module1_file']}")
        print(f"      Data Processing: {match['module2_func']:<25} in {match['module2_file']}")
else:
    print(f"\n3. No significant similar matches after filtering common functions")

print("\n" + "="*100)
print("This filtered information is useful for:")
print("  - Identifying truly unique shared functionality (not generic Python methods)")
print("  - Finding domain-specific code that might share similar vulnerabilities")
print("  - Understanding architectural similarities beyond common patterns")
print("  - Prioritizing security analysis on meaningful code patterns")
print("="*100)


SIMILAR FUNCTIONS ANALYSIS RESULTS (Excluding common function names)

1. Summary:
   - Total exact matches: 44
   - Filtered exact matches (excluding common names): 2
   - Total similar pairs: 50
   - Filtered similar pairs (excluding common names): 8

2. Exact Matches (first 10):
   1. Function: _encode
      LLM Core:        swift/llm/template/base.py
      Data Processing: src/llamafactory/data/template.py
   2. Function: _encode
      LLM Core:        swift/llm/template/base.py
      Data Processing: src/llamafactory/data/template.py

3. Similar Matches (first 10):
   1. Similarity: 0.7153
      LLM Core:        _align_blank_suffix       in swift/llm/infer/infer_engine/utils.py
      Data Processing: align_dataset             in src/llamafactory/data/converter.py
   2. Similarity: 0.7047
      LLM Core:        put                       in swift/llm/infer/infer_engine/utils.py
      Data Processing: apply                     in src/llamafactory/data/formatter.py
   3. Similarity: 0.

In [22]:
list(zip([1,2], [3,4]))### Example Usage: Compare Other Module Pairs

[(1, 3), (2, 4)]